In [36]:
import pandas as pd
import numpy as np
from math import log2

In [37]:
df = pd.read_csv("decisiontree1.csv")
df.head()

,age,income,student,credit_rating,buys_computer
0,<=30,high,no,fair,no
1,<=30,high,no,excellent,no
2,31...40,high,no,fair,yes
3,>40,medium,no,fair,yes
4,>40,low,yes,fair,yes


$$Info(D)=-\Sigma_{i=1}^mp_i\log_2(p_i)$$

In [38]:
def Entropy(D : pd.DataFrame) -> float:
    labels = {}
    for label in D[D.columns[-1]].unique():
        labels[label] = 0
    for label in D[D.columns[-1]]:
        for key in labels.keys():
            if label == key: labels[key] += 1
    tong = 0
    denominator = sum(labels.values())
    for label in labels:
        tong += - (labels[label] / denominator) * log2(labels[label] / denominator)
    return tong

$$Info_A(D)=\Sigma_{j=1}^v\frac{|D_j|}{|D|}\times Info(D_j)$$

In [39]:
def Info(D: pd.DataFrame, col: str) -> float:
    bags = {}
    
    for detail in D[col].unique(): bags[detail] = 0
    for detail in D[col]: bags[detail] += 1
    
    tong = 0
    denominator = sum(bags.values())
    for name in D[col].unique():
        new_df = D[D[col] == name]
        tong += bags[name] / denominator * Entropy(new_df)       
    return tong

$$Gain(A)=Info(D)-Info_A(D)$$

In [40]:
def Gain(D: pd.DataFrame, col: str):
    return Entropy(D) - Info(D, col)

# Implement

In [41]:
def DecisionTree(D: pd.DataFrame):
    if Entropy(D) == 0.0:
        print('reached leaf')
        return None
    
    label = D.columns[-1]
    f = 0.0
    fname = ""
    for feature in D.columns:
        if feature == label: continue
        
        if Gain(D, feature) > f:
            f = Gain(D, feature)
            fname = feature
    print('###################')
    print(fname)
    for item in D[fname].unique():
        new_df = D[D[fname] == item]
        new_df = new_df.drop(columns=fname).reset_index(drop=True)
        print('-----------------------------')
        print(item)
        print(new_df)
        DecisionTree(new_df)

DecisionTree(df)

###################
age
-----------------------------
<=30
   income student credit_rating buys_computer
0    high      no          fair            no
1    high      no     excellent            no
2  medium      no          fair            no
3     low     yes          fair           yes
4  medium     yes     excellent           yes
###################
student
-----------------------------
no
   income credit_rating buys_computer
0    high          fair            no
1    high     excellent            no
2  medium          fair            no
reached leaf
-----------------------------
yes
   income credit_rating buys_computer
0     low          fair           yes
1  medium     excellent           yes
reached leaf
-----------------------------
31...40
   income student credit_rating buys_computer
0    high      no          fair           yes
1     low     yes     excellent           yes
2  medium      no     excellent           yes
3    high     yes          fair           yes
reached le

Phần trên là tui làm, phần dưới là AI nâng cấp

In [42]:
new_data = {'age': '31..40', 'income': 'high', 'student': 'no', 'credit_rating':'excellent'}

In [43]:
import pandas as pd

def DecisionTree(D: pd.DataFrame):
    label = D.columns[-1]
    
    # 1. Điều kiện dừng: Nếu Entropy = 0 (tất cả data cùng 1 nhãn)
    if Entropy(D) == 0.0:
        # Trả về giá trị nhãn của dòng đầu tiên
        return D.iloc[0][label] 
    
    # Nếu không còn thuộc tính nào để chia nhánh ngoài cột label
    if len(D.columns) == 1:
        return D[label].mode()[0] # Trả về nhãn phổ biến nhất
    
    # 2. Tìm thuộc tính phân nhánh tốt nhất
    f = 0.0
    fname = ""
    for feature in D.columns:
        if feature == label: continue
        
        # Giả sử hàm Gain() đã được bạn định nghĩa
        current_gain = Gain(D, feature)
        if current_gain >= f:
            f = current_gain
            fname = feature
            
    # 3. Khởi tạo cây với Node gốc là thuộc tính tốt nhất
    tree = {fname: {}}
    
    # 4. Tạo các nhánh con
    for item in D[fname].unique():
        new_df = D[D[fname] == item]
        new_df = new_df.drop(columns=fname).reset_index(drop=True)
        
        # Đệ quy để xây dựng tiếp nhánh con và gán vào cây
        tree[fname][item] = DecisionTree(new_df)
        
    return tree

In [44]:
def predict(tree, data):
    # Nếu tree không phải là dictionary, tức là ta đã đi đến Node lá (chứa kết quả)
    if not isinstance(tree, dict):
        return tree
    
    # Lấy tên thuộc tính tại Node hiện tại (VD: 'age')
    root_node = next(iter(tree))
    
    # Lấy giá trị của thuộc tính đó trong dữ liệu mới (VD: '31..40')
    feature_value = data.get(root_node)
    
    # Kiểm tra xem giá trị này có nhánh đi tiếp trong cây không
    if feature_value in tree[root_node]:
        # Tiếp tục đệ quy đi xuống nhánh con
        return predict(tree[root_node][feature_value], data)
    else:
        # Xử lý ngoại lệ: Nếu dữ liệu mới chứa giá trị chưa từng gặp khi train
        return "Không thể dự đoán (Dữ liệu ngoại lệ)"

In [46]:
def visualize_tree(tree, indent=""):
    """
    Hàm đệ quy để in cây quyết định dạng Dictionary ra màn hình.
    """
    # Nếu truyền vào một Node lá (chỉ là kết quả, không phải dict), in ra luôn
    if not isinstance(tree, dict):
        print(f" ➔ Dự đoán: {tree}")
        return

    # Lấy tên thuộc tính phân nhánh tại Node hiện tại
    root_node = next(iter(tree))
    branches = tree[root_node]
    keys = list(branches.keys())
    
    # Duyệt qua từng giá trị của thuộc tính để vẽ nhánh
    for i, key in enumerate(keys):
        is_last = (i == len(keys) - 1)
        # Sử dụng các ký tự ASCII để vẽ nhánh
        prefix = "└── " if is_last else "├── "
        
        # In ra điều kiện của nhánh
        print(f"{indent}{prefix}{root_node} == '{key}'", end="")
        
        subtree = branches[key]
        
        # Nếu nhánh con đi tiếp vào một thuộc tính khác (dict)
        if isinstance(subtree, dict):
            print() # Xuống dòng
            next_indent = indent + ("    " if is_last else "│   ")
            visualize_tree(subtree, next_indent)
        # Nếu nhánh con là kết quả cuối cùng (lá)
        else:
            print(f" ➔ Dự đoán: [{subtree}]")

# ---------------------------------------------------------
# CÁCH SỬ DỤNG
# (Giả sử trained_tree là cây bạn đã huấn luyện từ bước trước)
# 
# trained_tree = DecisionTree(df)
# print("SƠ ĐỒ CÂY QUYẾT ĐỊNH:")
# visualize_tree(trained_tree)

In [47]:
# 1. Xây dựng cây từ DataFrame df (huấn luyện)
trained_tree = DecisionTree(df)
print("Cấu trúc cây:", trained_tree)
# Output ví dụ: {'age': {'<=30': {'student': {'no': 'no', 'yes': 'yes'}}, '31..40': 'yes', '>40': {...}}}

# 2. Khai báo dữ liệu mới
new_data = {'age': '31...40', 'income': 'high', 'student': 'no', 'credit_rating': 'excellent'}

# 3. Đưa vào dự đoán
result = predict(trained_tree, new_data)
print("Kết quả dự đoán:", result)
visualize_tree(trained_tree)

Cấu trúc cây: {'age': {'<=30': {'student': {'no': 'no', 'yes': 'yes'}}, '31...40': 'yes', '>40': {'credit_rating': {'fair': 'yes', 'excellent': 'no'}}}}
Kết quả dự đoán: yes
├── age == '<=30'
│   ├── student == 'no' ➔ Dự đoán: [no]
│   └── student == 'yes' ➔ Dự đoán: [yes]
├── age == '31...40' ➔ Dự đoán: [yes]
└── age == '>40'
    ├── credit_rating == 'fair' ➔ Dự đoán: [yes]
    └── credit_rating == 'excellent' ➔ Dự đoán: [no]
